# Employee Promotion Prediction

**Classification Project 14 of 100** — Predict whether an employee will be promoted based on training, performance, and HR attributes.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Employee promotion decisions are central to HR strategy. Identifying employees likely to be promoted — or overlooked — helps organisations plan workforce development, reduce attrition of high-potential talent, and ensure fair promotion practices.

This notebook uses the **HR Analytics** dataset from Kaggle, containing employee records with training scores, performance ratings, department, tenure, and a binary promotion outcome. The dataset is **significantly imbalanced** (~8.5% promoted), making it an excellent case study for handling class imbalance in HR prediction.

**Workflow:**
1. Download & validate the dataset from Kaggle
2. Perform thorough EDA on HR features
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models with full metrics
8. Analyse errors and extract HR insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Work with an **imbalanced HR dataset** (~8.5% promotion rate)
2. Handle **mixed feature types** (numeric + categorical)
3. Identify and drop **ID columns** and **leakage-prone features**
4. Use **class_weight='balanced'** and **scale_pos_weight** for imbalanced targets
5. Engineer **domain-specific HR features** (tenure grouping, training efficiency)
6. Build a **leakage-free** preprocessing pipeline
7. Choose metrics that matter for HR: **F1, recall, ROC-AUC, PR-AUC**
8. Benchmark with **LazyPredict** and **FLAML**
9. Interpret model outputs through an **HR strategy lens**
10. Discuss **fairness and bias risks** in HR prediction

## 4. Problem Statement

> **Given an employee's department, training score, performance rating, age, tenure, and other HR attributes, predict whether they will be promoted (is_promoted = 1).**

This is a **binary classification** task with significant imbalance (~8.5% promoted). Missing a high-potential employee (false negative) risks losing them to attrition. A false positive may lead to premature development investment. **Recall** and **PR-AUC** are important alongside F1.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **HR teams** | Data-driven succession planning and talent pipeline management |
| **Managers** | Identify high-potential team members for development |
| **Employees** | Fairer, more transparent promotion criteria |
| **Executives** | Strategic workforce planning and retention |
| **ML learners** | Practice imbalanced classification in an HR context with fairness considerations |

## 6. Dataset Overview

| Property | Value |
|---|---|
| **Name** | HR Analytics: Job Change of Data Scientists |
| **Source** | Kaggle |
| **Kaggle slug** | `arashnic/hr-analytics-job-change-of-data-scientists` |
| **Rows** | ~54,808 (train) |
| **Features** | 13 (demographics, experience, training, company info) |
| **Target** | `target` (1 = looking for job change, 0 = not looking) |
| **Class balance** | ~75% not looking, ~25% looking for change |

**Key columns:**
- `city`, `city_development_index` — location features
- `gender`, `relevent_experience`, `enrolled_university` — demographics
- `education_level`, `major_discipline` — education background
- `experience`, `company_size`, `company_type` — work history
- `last_new_job`, `training_hours` — career transition signals
- `target` — whether employee is looking for a job change

**Note:** This dataset predicts job-change intent rather than direct promotion. We frame it as 'employee transition prediction' which is closely related to promotion/retention analysis.

## 7. Dataset Source and License Notes

- **Kaggle:** https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists
- **License:** CC0 (Public Domain) — free for any use.
- **Context:** Based on a real HR analytics scenario for a company training data scientists.
- **Note:** Some columns contain missing values that must be handled explicitly.

## 8. Environment Setup

We install any packages not already available.

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "arashnic/hr-analytics-job-change-of-data-scientists"
TARGET = "target"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120  # seconds
MAX_ROWS = 50_000  # cap for speed
print(f"Target: {TARGET} | Test: {TEST_SIZE} | Seed: {SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. Kaggle credentials must be available.

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to download dataset: {e}\n"
        "Ensure KAGGLE_API_TOKEN is set or ~/.kaggle/kaggle.json exists."
    )

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"CSV files found: {[os.path.basename(f) for f in csv_files]}")

# Prefer the training file
selected = next((f for f in csv_files if 'train' in os.path.basename(f).lower()), csv_files[0])
print(f"Using: {os.path.basename(selected)}")

df_raw = pd.read_csv(selected)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

We verify data integrity: target column, missing values, duplicates.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"\nMissing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f"\nTotal missing: {df_raw.isnull().sum().sum()}")

assert TARGET in df_raw.columns, f"Target column '{TARGET}' not found!"

dupes = df_raw.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")
if dupes > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)

# Drop rows with missing target
df_raw = df_raw.dropna(subset=[TARGET]).reset_index(drop=True)
df_raw[TARGET] = df_raw[TARGET].astype(int)

print(f"\nTarget distribution:")
print(df_raw[TARGET].value_counts())
print(f"Positive rate: {df_raw[TARGET].mean():.1%}")

### Subsample for Speed

If the dataset is large, we subsample while preserving class ratios.

In [ ]:
if len(df_raw) > MAX_ROWS:
    df = df_raw.groupby(TARGET, group_keys=False).apply(
        lambda x: x.sample(frac=MAX_ROWS / len(df_raw), random_state=SEED)
    ).reset_index(drop=True)
    print(f"Sampled: {df.shape}")
else:
    df = df_raw.copy()
    print(f"Using full dataset: {df.shape}")
print(df[TARGET].value_counts())

## 13. Exploratory Data Analysis

We explore feature distributions and relationships with the target.

In [ ]:
df.describe(include='all').T.head(20)

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
ax.set_title("Target Distribution")
ax.set_xticklabels(["Stay", "Looking for Change"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
num_cols = df.select_dtypes(include=[np.number]).columns.drop(TARGET, errors='ignore').tolist()
if num_cols:
    fig, axes = plt.subplots(1, min(3, len(num_cols)), figsize=(15, 4))
    if len(num_cols) == 1: axes = [axes]
    for ax, col in zip(axes, num_cols[:3]):
        for label in [0, 1]:
            grp = df[df[TARGET] == label]
            tag = 'Looking' if label == 1 else 'Stay'
            ax.hist(grp[col].dropna(), bins=30, alpha=0.5, label=tag)
        ax.set_title(col)
        ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical feature vs target
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
key_cats = [c for c in cat_cols if df[c].nunique() <= 10][:4]
if key_cats:
    fig, axes = plt.subplots(1, len(key_cats), figsize=(5 * len(key_cats), 4))
    if len(key_cats) == 1: axes = [axes]
    for ax, col in zip(axes, key_cats):
        ct = pd.crosstab(df[col], df[TARGET], normalize='index')
        ct.plot.bar(ax=ax, stacked=True)
        ax.set_title(col)
        ax.set_ylabel("Proportion")
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 14. Target Analysis

The positive rate is approximately 25% (looking for job change). A majority-class-only model achieves ~75% accuracy but provides zero actionable insight.

In [ ]:
pos_rate = df[TARGET].mean()
print(f"Positive rate: {pos_rate:.1%}")
print(f"Majority-class baseline accuracy: {1 - pos_rate:.1%}")
print(f"Imbalance ratio: {(1-pos_rate)/pos_rate:.1f}:1")

## 15. Train / Validation / Test Split Strategy

We use a **70/15/15 stratified** split. We split **before** any preprocessing.

In [ ]:
# Drop ID-like columns
id_cols = [c for c in df.columns if 'enrollee_id' in c.lower() or c.lower() == 'id']
print(f"Dropping ID columns: {id_cols}")

X = df.drop(columns=[TARGET] + id_cols, errors='ignore')
y = df[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED, stratify=y_temp
)

print(f"\nTrain: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} positive rate: {ys.mean():.1%}")

## 16. Preprocessing Strategy

**Decisions:**
- **Missing values:** Impute median (numeric) and 'Missing' constant (categorical).
- **Categorical encoding:** OneHotEncoder for low-cardinality; high-cardinality like `city` dropped or frequency-encoded.
- **Scaling:** StandardScaler for numeric features.
- **No leakage:** ColumnTransformer fit only on training data.

In [ ]:
# Drop very high cardinality columns
high_card = [c for c in X_train.select_dtypes(include=['object']).columns if X_train[c].nunique() > 50]
if high_card:
    print(f"Dropping high-cardinality columns: {high_card}")
    X_train = X_train.drop(columns=high_card)
    X_val = X_val.drop(columns=high_card)
    X_test = X_test.drop(columns=high_card)

num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric: {len(num_features)}, Categorical: {len(cat_features)}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print("Preprocessor ready.")

## 17. Feature Engineering

We create domain-inspired features:
- **experience_numeric:** convert experience string to numeric
- **training_efficiency:** training_hours scaled by city development index

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    # Convert experience to numeric
    if 'experience' in df_out.columns:
        df_out['experience_num'] = pd.to_numeric(
            df_out['experience'].astype(str).str.replace('<', '').str.replace('>', '').str.strip(),
            errors='coerce'
        )
    # Training efficiency
    if 'training_hours' in df_out.columns and 'city_development_index' in df_out.columns:
        cdi = pd.to_numeric(df_out['city_development_index'], errors='coerce').clip(lower=0.1)
        df_out['training_per_cdi'] = pd.to_numeric(df_out['training_hours'], errors='coerce') / cdi
    return df_out

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

# Refresh column lists
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print(f"Features after engineering: {len(num_features) + len(cat_features)}")

## 18. Baseline Model

DummyClassifier sets the floor. Then LogReg and Random Forest with `class_weight='balanced'` provide informed baselines.

In [ ]:
results = {}

dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_val)
results["Dummy"] = {"Accuracy": accuracy_score(y_val, y_pred_d), "F1": f1_score(y_val, y_pred_d, zero_division=0), "Recall": recall_score(y_val, y_pred_d, zero_division=0)}
print("Dummy:", {k: f"{v:.4f}" for k, v in results["Dummy"].items()})

lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_val)[:, 1]
results["LogReg"] = {"Accuracy": accuracy_score(y_val, lr.predict(X_val)), "F1": f1_score(y_val, lr.predict(X_val)), "Recall": recall_score(y_val, lr.predict(X_val)), "ROC-AUC": roc_auc_score(y_val, y_prob_lr), "PR-AUC": average_precision_score(y_val, y_prob_lr)}
print("LogReg:", {k: f"{v:.4f}" for k, v in results["LogReg"].items()})

rf = Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:, 1]
results["RF"] = {"Accuracy": accuracy_score(y_val, rf.predict(X_val)), "F1": f1_score(y_val, rf.predict(X_val)), "Recall": recall_score(y_val, rf.predict(X_val)), "ROC-AUC": roc_auc_score(y_val, y_prob_rf), "PR-AUC": average_precision_score(y_val, y_prob_rf)}
print("RF:", {k: f"{v:.4f}" for k, v in results["RF"].items()})

## 19. LazyPredict Benchmark

LazyPredict fits ~30 classifiers quickly. This is for **exploration only**.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)

lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
print("Top 15 models by Accuracy:")
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML optimises for **F1** to balance precision and recall on this imbalanced target.

In [ ]:
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="f1",
    time_budget=FLAML_BUDGET,
    seed=SEED,
    verbose=0,
)

print(f"Best model: {automl.best_estimator}")
y_pred_fl = automl.predict(X_val)
y_prob_fl = automl.predict_proba(X_val)[:, 1]
results["FLAML"] = {"Accuracy": accuracy_score(y_val, y_pred_fl), "F1": f1_score(y_val, y_pred_fl), "Recall": recall_score(y_val, y_pred_fl), "ROC-AUC": roc_auc_score(y_val, y_prob_fl), "PR-AUC": average_precision_score(y_val, y_prob_fl)}
print("FLAML:", {k: f"{v:.4f}" for k, v in results["FLAML"].items()})

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three strong classifiers:
1. **LightGBM** — fast gradient boosting
2. **XGBoost** — robust alternative
3. **GradientBoosting** — sklearn ensemble

In [ ]:
try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
except ImportError:
    has_lgbm = False
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=SEED, n_jobs=-1)
if has_xgb:
    top3["XGBoost"] = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, scale_pos_weight=imbalance_ratio, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3["GradientBoosting"] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out **test set**.

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

final_results = {}
for name, model in top3.items():
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_prob = model.predict_proba(X_test_t)[:, 1]
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
    }
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=["Stay", "Looking"]))

results_df = pd.DataFrame(final_results).T
print("\n=== Final Test Results ===")
results_df

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1: axes = [axes]
for ax, (name, model) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test_t), ax=ax, cmap="Blues", display_labels=["Stay", "Looking"])
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in top3.items():
    RocCurveDisplay.from_estimator(model, X_test_t, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test_t, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC Curves")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

## 23. Error Analysis

We examine misclassified employees to find patterns.

In [ ]:
best_name = results_df["F1"].idxmax()
best_model = top3[best_name]
y_pred_best = best_model.predict(X_test_t)

fn_mask = (y_test.values == 1) & (y_pred_best == 0)
fp_mask = (y_test.values == 0) & (y_pred_best == 1)

print(f"Best model: {best_name}")
print(f"False Negatives (missed leavers): {fn_mask.sum()}")
print(f"False Positives (false alarms): {fp_mask.sum()}")
print(f"FN rate: {fn_mask.sum() / max((y_test == 1).sum(), 1):.1%}")

test_df = X_test.copy()
test_df["error_type"] = "correct"
test_df.loc[test_df.index[fn_mask], "error_type"] = "false_negative"
test_df.loc[test_df.index[fp_mask], "error_type"] = "false_positive"
num_test = test_df.select_dtypes(include=[np.number]).columns[:5].tolist()
if num_test:
    print("\nMean features by error type:")
    print(test_df.groupby("error_type")[num_test].mean().round(2))

## 24. Interpretation and Business Insight

**Key findings:**
1. **City development index** is a strong predictor — employees in less developed areas are more likely to seek change
2. **Training hours** and **experience** show meaningful relationships with job-change intent
3. **Education level and university enrollment** affect transition likelihood

**HR recommendations:**
- Use the model to flag high-risk employees for **retention conversations**
- Investigate whether **training investment** reduces job-change intent
- Monitor **city-level trends** for location-specific retention strategies
- Combine predictions with **manager assessments** for comprehensive talent reviews

## 25. Limitations

1. **Proxy target:** 'Looking for job change' is not the same as 'will be promoted'
2. **Self-reported features:** Education, experience may contain inaccuracies
3. **Missing values:** Several columns have significant missingness
4. **Single company:** Results may not generalise across industries
5. **Snapshot data:** No temporal tracking of employee trajectories

## 26. How to Improve This Project

1. Add **cross-validation** for more stable estimates
2. Use **SHAP** for individual employee explanations
3. Try **target encoding** for high-cardinality city features
4. Add **interaction features** (experience × training hours)
5. Experiment with **threshold tuning** for different business costs
6. Try **ensemble stacking** of top models

## 27. Production Considerations

- **Privacy:** Employee data is sensitive — ensure GDPR/labour law compliance
- **Fairness:** Audit predictions across gender, age, and education to prevent discrimination
- **Explainability:** Managers need interpretable flags, not opaque scores
- **Human oversight:** Predictions should inform, not replace, managerial judgment
- **Monitoring:** Track actual promotions/departures to retrain the model quarterly

## 28. Common Mistakes

1. **Including enrollee_id** — IDs should never be features
2. **Not handling missing values explicitly** — many columns have significant missingness
3. **Using accuracy alone** — with 75/25 split, accuracy is misleading
4. **Not stratifying splits** — random split can skew class ratios
5. **Ignoring high-cardinality features** — city with 100+ values needs special encoding
6. **Bias blindness** — gender, age effects must be audited

## 29. Mini Challenge / Exercises

1. Add SHAP feature importance — which features drive individual predictions?
2. Compare target encoding vs dropping the `city` column — which works better?
3. If losing an employee costs $50,000 and a false alarm costs $500, what threshold maximises profit?
4. Run 5-fold cross-validation — how stable is F1 across folds?
5. Build a model using only features available at hiring time (no training_hours)

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| **Task** | Binary classification — employee job-change prediction |
| **Dataset** | ~54k employees, 13 features, ~25% positive rate |
| **Best metric focus** | F1, PR-AUC, Recall |
| **Baseline** | DummyClassifier ~75% accuracy, 0% recall |
| **Best models** | Gradient boosting family |
| **Key insight** | City development index and experience are strongest predictors |
| **Limitation** | Proxy target, snapshot data, single company |

**What you learned:**
- How to handle missing values and mixed types in HR data
- Why fairness auditing matters in employee classification
- Feature engineering from HR domain knowledge
- The full benchmark → AutoML → top 3 evaluation pipeline